In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Load original dataset
df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")

print("Original dataset shape:", df.shape)
df.head()

Original dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
print("Missing values:")
print(df.isnull().sum())

print("\nBlank TotalCharges:")
print(
    df["TotalCharges"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

Missing values:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Blank TotalCharges:
11


In [4]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

print("Missing TotalCharges after conversion:",
      df["TotalCharges"].isnull().sum())

Missing TotalCharges after conversion: 11


In [5]:
blank_rows = df[df["TotalCharges"].isnull()]

print("Rows with missing TotalCharges:", len(blank_rows))

print(blank_rows[["tenure", "TotalCharges"]].head(15))

print(
    "All missing TotalCharges have zero tenure:",
    blank_rows["tenure"].eq(0).all()
)

Rows with missing TotalCharges: 11
      tenure  TotalCharges
488        0           NaN
753        0           NaN
936        0           NaN
1082       0           NaN
1340       0           NaN
3331       0           NaN
3826       0           NaN
4380       0           NaN
5218       0           NaN
6670       0           NaN
6754       0           NaN
All missing TotalCharges have zero tenure: True


In [6]:
if blank_rows["tenure"].eq(0).all():
    df["TotalCharges"] = df["TotalCharges"].fillna(0)
else:
    raise ValueError(
        "Investigate missing TotalCharges before filling."
    )

print("Remaining missing values:", df.isnull().sum().sum())

Remaining missing values: 0


In [7]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [8]:
df = df.drop_duplicates().copy()

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (7043, 21)


In [9]:
df = df.drop(columns=["customerID"])

print("Shape after removing customerID:", df.shape)

Shape after removing customerID: (7043, 20)


In [10]:
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print(df["Churn"].value_counts())

Churn
0    5174
1    1869
Name: count, dtype: int64


In [11]:
print(df["Churn"].isnull().sum())
print(df["Churn"].unique())

0
[0 1]


In [12]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeature columns:")
print(X.columns.tolist())

Features shape: (7043, 19)
Target shape: (7043,)

Feature columns:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (5634, 19)
Testing features: (1409, 19)
Training target: (5634,)
Testing target: (1409,)


In [14]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


C:\Users\navka\AppData\Local\Temp\ipykernel_25572\1124402278.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [15]:
# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [16]:
# Fit only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data using the already fitted preprocessor
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (5634, 45)
Processed testing shape: (1409, 45)


In [17]:
feature_names = preprocessor.get_feature_names_out()

print("Total processed features:", len(feature_names))

print("\nFirst 20 processed features:")
print(feature_names[:20])

Total processed features: 45

First 20 processed features:
['num__SeniorCitizen' 'num__tenure' 'num__MonthlyCharges'
 'num__TotalCharges' 'cat__gender_Female' 'cat__gender_Male'
 'cat__Partner_No' 'cat__Partner_Yes' 'cat__Dependents_No'
 'cat__Dependents_Yes' 'cat__PhoneService_No' 'cat__PhoneService_Yes'
 'cat__MultipleLines_No' 'cat__MultipleLines_No phone service'
 'cat__MultipleLines_Yes' 'cat__InternetService_DSL'
 'cat__InternetService_Fiber optic' 'cat__InternetService_No'
 'cat__OnlineSecurity_No' 'cat__OnlineSecurity_No internet service']


In [18]:
print("Training data contains NaN:",
      np.isnan(X_train_processed).any())

print("Testing data contains NaN:",
      np.isnan(X_test_processed).any())

Training data contains NaN: False
Testing data contains NaN: False


In [19]:
def convert_to_array(data):
    if hasattr(data, "toarray"):
        return data.toarray()
    return np.asarray(data)

In [20]:
X_train_array = convert_to_array(X_train_processed)
X_test_array = convert_to_array(X_test_processed)

y_train_array = y_train.to_numpy()
y_test_array = y_test.to_numpy()

In [21]:
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

# Save processed datasets
np.save("../data/processed/X_train.npy", X_train_array)
np.save("../data/processed/X_test.npy", X_test_array)

np.save("../data/processed/y_train.npy", y_train_array)
np.save("../data/processed/y_test.npy", y_test_array)

# Save preprocessing pipeline
joblib.dump(
    preprocessor,
    "../models/preprocessor.joblib"
)

# Save cleaned dataset for reference
df.to_csv(
    "../data/processed/cleaned_telco_churn.csv",
    index=False
)

# Save feature names
np.save(
    "../data/processed/feature_names.npy",
    feature_names
)

print("All processed datasets saved successfully!")

All processed datasets saved successfully!


In [22]:
print("========== RETAINAI DATA PREPARATION ==========")

print("Training records:", len(X_train_array))
print("Testing records:", len(X_test_array))

print("Training features:", X_train_array.shape[1])
print("Testing features:", X_test_array.shape[1])

print("\nTraining churn distribution:")
print(pd.Series(y_train_array).value_counts())

print("\nTesting churn distribution:")
print(pd.Series(y_test_array).value_counts())

print("\nMissing values in training:",
      np.isnan(X_train_array).sum())

print("Missing values in testing:",
      np.isnan(X_test_array).sum())

print("\nPHASE 3 DATA PREPARATION COMPLETED!")

========== RETAINAI DATA PREPARATION ==========
Training records: 5634
Testing records: 1409
Training features: 45
Testing features: 45

Training churn distribution:
0    4139
1    1495
Name: count, dtype: int64

Testing churn distribution:
0    1035
1     374
Name: count, dtype: int64

Missing values in training: 0
Missing values in testing: 0

PHASE 3 DATA PREPARATION COMPLETED!
